In [1]:
import pandas as pd
import requests


url = "https://apisidra.ibge.gov.br/values/t/7060/n1/all/v/63/p/all/c315/all?formato=json"
r = requests.get(url)
data = r.json()

df = pd.DataFrame(data[1:])  # pula o cabeçalho


In [2]:
import sys, os
project_root = os.path.abspath("..")  # sobe um nível
sys.path.append(project_root)
print(project_root)

from src.api.sidra_client import sidra_query
from src.pipelines.ipca import codigos as cod_mod
from src.api.sidra_client import sidra_query
from src.pipelines.ipca.base import get_ipca_nivel

/home/moacir/projects/ml/analytics/inflation


In [3]:
df.head()

,NC,NN,MC,MN,V,D1C,D1N,D2C,D2N,D3C,D3N,D4C,D4N
0,1,Brasil,2,%,0.21,1,Brasil,63,IPCA - Variação mensal,202001,janeiro 2020,7169,Índice geral
1,1,Brasil,2,%,0.39,1,Brasil,63,IPCA - Variação mensal,202001,janeiro 2020,7170,1.Alimentação e bebidas
2,1,Brasil,2,%,0.20,1,Brasil,63,IPCA - Variação mensal,202001,janeiro 2020,7171,11.Alimentação no domicílio
3,1,Brasil,2,%,0.75,1,Brasil,63,IPCA - Variação mensal,202001,janeiro 2020,7172,"1101.Cereais, leguminosas e oleaginosas"
4,1,Brasil,2,%,1.31,1,Brasil,63,IPCA - Variação mensal,202001,janeiro 2020,7173,1101002.Arroz


In [4]:
df_indice = sidra_query(7060, 63, "all", {"315": "all"})
df_indice.columns


2026-07-26 21:58:17,667 | INFO | src.api.sidra_client | Consultando SIDRA: tabela=7060 | → variavel=63 | periodo=all
2026-07-26 21:58:21,595 | INFO | src.api.sidra_client | Pipeline para conversão de valores concluído com sucesso.


Index(['NC', 'NN', 'MC', 'MN', 'valor', 'D1C', 'D1N', 'D2C', 'D2N', 'periodo',
       'D3N', 'categoria', 'categoria_nome'],
      dtype='str')

In [3]:
df_grupos = get_ipca_nivel(1)
df_grupos.head()
len(df_grupos)


2026-07-26 22:06:07,135 | INFO | src.api.sidra_client | Consultando SIDRA: tabela=7060 | → variavel=63 | periodo=all
2026-07-26 22:06:08,658 | INFO | src.api.sidra_client | Pipeline para conversão de valores concluído com sucesso.
2026-07-26 22:06:08,680 | INFO | src.api.sidra_client | Consultando SIDRA: tabela=7060 | → variavel=66 | periodo=all
2026-07-26 22:06:11,668 | INFO | src.api.sidra_client | Pipeline para conversão de valores concluído com sucesso.
2026-07-26 22:06:11,687 | INFO | src.api.sidra_client | Consultando SIDRA: tabela=7060 | → variavel=2265 | periodo=all
2026-07-26 22:06:14,569 | INFO | src.api.sidra_client | Pipeline para conversão de valores concluído com sucesso.


702

In [ ]:

# garantir D4C numérico
df["D4C"] = pd.to_numeric(df["D4C"], errors="coerce").astype("Int64")
df = df.dropna(subset=["D4C"])
df["D4C"] = df["D4C"].astype(int)

codigos = df[["D4C", "D4N"]].drop_duplicates().sort_values("D4C")

codigos["prefix"] = codigos["D4N"].str.split(".").str[0]
codigos["prefix_len"] = codigos["prefix"].str.len()

cod_geral = codigos[codigos["D4N"] == "Índice geral"]["D4C"].tolist()
cod_grupos = codigos[codigos["prefix_len"] == 1]["D4C"].tolist()
cod_subgrupos = codigos[codigos["prefix_len"] == 2]["D4C"].tolist()
cod_itens = codigos[codigos["prefix_len"] == 4]["D4C"].tolist()
cod_subitens = codigos[codigos["prefix_len"] == 7]["D4C"].tolist()


In [34]:
df_indice = sidra_query(
    tabela=7060,
    variavel=63,
    periodo="all",
    classificacao={"315": "all"},
)

df_indice.head()
df_indice["categoria"].nunique(), df_indice["periodo"].nunique()

2026-07-26 21:05:36,685 | INFO | src.api.sidra_client | Consultando SIDRA: tabela=7060 | → variavel=63 | periodo=all
2026-07-26 21:05:41,003 | INFO | src.api.sidra_client | Pipeline para conversão de valores concluído com sucesso.


(457, 78)

In [36]:
df_grupos = get_ipca_nivel(cod_grupos, periodo="all")
df_grupos.head()
len(df_grupos)

2026-07-26 21:06:34,521 | INFO | src.api.sidra_client | Consultando SIDRA: tabela=7060 | → variavel=63 | periodo=all
2026-07-26 21:06:36,512 | INFO | src.api.sidra_client | Pipeline para conversão de valores concluído com sucesso.
2026-07-26 21:06:36,527 | INFO | src.api.sidra_client | Consultando SIDRA: tabela=7060 | → variavel=66 | periodo=all
2026-07-26 21:06:41,172 | INFO | src.api.sidra_client | Pipeline para conversão de valores concluído com sucesso.
2026-07-26 21:06:41,186 | INFO | src.api.sidra_client | Consultando SIDRA: tabela=7060 | → variavel=2265 | periodo=all
2026-07-26 21:06:42,685 | INFO | src.api.sidra_client | Pipeline para conversão de valores concluído com sucesso.


0

In [37]:
df_indice = sidra_query(
    tabela=7060,
    variavel=63,
    periodo="all",
    classificacao={"315": "all"},
)

df_indice["categoria"].unique()[:50]
len(df_indice["categoria"].unique())


2026-07-26 21:10:06,005 | INFO | src.api.sidra_client | Consultando SIDRA: tabela=7060 | → variavel=63 | periodo=all
2026-07-26 21:10:09,611 | INFO | src.api.sidra_client | Pipeline para conversão de valores concluído com sucesso.


457

In [38]:
set(cod_grupos) - set(df_indice["categoria"].unique())


{7170, 7445, 7486, 7558, 7625, 7660, 7712, 7766, 7786}

In [39]:
codigos[["D4C", "D4N", "prefix", "prefix_len"]].head(20)


,D4C,D4N,prefix,prefix_len
0,7169,Índice geral,Índice geral,12
1,7170,1.Alimentação e bebidas,1,1
2,7171,11.Alimentação no domicílio,11,2
3,7172,"1101.Cereais, leguminosas e oleaginosas",1101,4
4,7173,1101002.Arroz,1101002,7
5,7175,1101051.Feijão - mulatinho,1101051,7
6,7176,1101052.Feijão - preto,1101052,7
10,7184,"1102.Farinhas, féculas e massas",1102,4
11,7185,1102001.Farinha de arroz,1102001,7
12,7187,1102006.Macarrão,1102006,7


In [41]:
set(cod_grupos) & set(df_indice["categoria"].unique())


set()

In [42]:
set(cod_subgrupos) & set(df_indice["categoria"].unique())


set()

In [43]:
# Garantir que D4C é inteiro
codigos["D4C"] = pd.to_numeric(codigos["D4C"], errors="coerce").astype("Int64")
codigos = codigos.dropna(subset=["D4C"])
codigos["D4C"] = codigos["D4C"].astype(int)

# Recriar prefixos
codigos["prefix"] = codigos["D4N"].str.split(".").str[0]
codigos["prefix_len"] = codigos["prefix"].str.len()

# Recriar listas
cod_grupos = codigos[codigos["prefix_len"] == 1]["D4C"].tolist()
cod_subgrupos = codigos[codigos["prefix_len"] == 2]["D4C"].tolist()
cod_itens = codigos[codigos["prefix_len"] == 4]["D4C"].tolist()
cod_subitens = codigos[codigos["prefix_len"] == 7]["D4C"].tolist()


In [44]:
set(cod_grupos) & set(df_indice["categoria"].unique())


set()

In [19]:
codigos = df[["D4C", "D4N"]].drop_duplicates()

codigos["prefix"] = codigos["D4N"].str.split(".").str[0]
codigos["prefix_len"] = codigos["prefix"].str.len()

codigos.head(20)

,D4C,D4N,prefix,prefix_len
0,7169,Índice geral,Índice geral,12
1,7170,1.Alimentação e bebidas,1,1
2,7171,11.Alimentação no domicílio,11,2
3,7172,"1101.Cereais, leguminosas e oleaginosas",1101,4
4,7173,1101002.Arroz,1101002,7
5,7175,1101051.Feijão - mulatinho,1101051,7
6,7176,1101052.Feijão - preto,1101052,7
7,47617,1101053.Feijão - macáçar (fradinho),1101053,7
8,12222,1101073.Feijão - carioca (rajado),1101073,7
9,47618,1101079.Milho (em grão),1101079,7


In [23]:
codigos = df[["D4C", "D4N"]].drop_duplicates()

codigos["prefix"] = codigos["D4N"].str.split(".").str[0]
codigos["prefix_len"] = codigos["prefix"].str.len()

cod_grupos = codigos[codigos["prefix_len"] == 1]["D4C"].tolist()
cod_subgrupos = codigos[codigos["prefix_len"] == 2]["D4C"].tolist()
cod_itens = codigos[codigos["prefix_len"] == 4]["D4C"].tolist()
cod_subitens = codigos[codigos["prefix_len"] == 7]["D4C"].tolist()


In [17]:
codigos = df[["D4C", "D4N"]].drop_duplicates()

codigos["prefix"] = codigos["D4N"].str.split(".").str[0]
codigos["prefix_len"] = codigos["prefix"].str.len()

codigos.head(20)

,D4C,D4N,prefix,prefix_len
0,7169,Índice geral,Índice geral,12
1,7170,1.Alimentação e bebidas,1,1
2,7171,11.Alimentação no domicílio,11,2
3,7172,"1101.Cereais, leguminosas e oleaginosas",1101,4
4,7173,1101002.Arroz,1101002,7
5,7175,1101051.Feijão - mulatinho,1101051,7
6,7176,1101052.Feijão - preto,1101052,7
7,47617,1101053.Feijão - macáçar (fradinho),1101053,7
8,12222,1101073.Feijão - carioca (rajado),1101073,7
9,47618,1101079.Milho (em grão),1101079,7


In [ ]:
from src.pipelines.ipca.base import get_ipca_nivel

df_test_grupos = get_ipca_nivel(cod_grupos, periodo="all")
df_test_grupos.head()
len(df_test_grupos)


In [9]:
codigos = df[["D4C", "D4N"]].drop_duplicates().sort_values("D4C")
codigos.head()


,D4C,D4N
85,101448,1107097.Capa de filé
102,101466,1108072.Peixe - curimatã
381,101642,6301004.Produto para barba
382,101644,6301006.Produto para pele
17,107608,1102029.Massa semipreparada


In [10]:
print(codigos["D4C"].min(), codigos["D4C"].max())
print(codigos["D4C"].unique()[:50])


101448 8874
<ArrowStringArray>
['101448', '101466', '101642', '101644', '107608', '107609', '107611',
 '107615', '107616', '107617', '107618', '107619', '107621', '107624',
 '107625', '107628', '107630', '107633', '107638', '107639', '107640',
 '107641', '107642', '107643', '107645', '107648', '107649', '107650',
 '107652', '107653', '107654', '107656', '107657', '107659', '107661',
 '107666', '107668', '107671', '107672', '107673', '107674', '107676',
 '107678', '107679', '107681', '107682', '107683', '107688', '107702',
 '109463']
Length: 50, dtype: str


In [55]:
url = "https://apisidra.ibge.gov.br/values/t/1737/n1/all/v/all/p/all?formato=json"
df = pd.DataFrame(requests.get(url).json()[1:])
#df.head()
df[["MC","MN","D2C","D2N"]].drop_duplicates()

,MC,MN,D2C,D2N
0,30,Número-índice,2266,IPCA - Número-índice (base: dezembro de 1993 =...
559,,,63,IPCA - Variação mensal
560,2,%,63,IPCA - Variação mensal
1118,,,2263,IPCA - Variação acumulada em 3 meses
1121,2,%,2263,IPCA - Variação acumulada em 3 meses
1677,,,2264,IPCA - Variação acumulada em 6 meses
1683,2,%,2264,IPCA - Variação acumulada em 6 meses
2236,,,69,IPCA - Variação acumulada no ano
2237,2,%,69,IPCA - Variação acumulada no ano
2795,,,2265,IPCA - Variação acumulada em 12 meses


In [56]:
import requests
import pandas as pd

url = "https://apisidra.ibge.gov.br/values/t/1737/n1/1/v/2266/p/all?formato=json"
df = pd.DataFrame(requests.get(url).json()[1:])
df.head()


,NC,NN,MC,MN,V,D1C,D1N,D2C,D2N,D3C,D3N
0,1,Brasil,30,Número-índice,0.0000000076183,1,Brasil,2266,IPCA - Número-índice (base: dezembro de 1993 =...,197912,dezembro 1979
1,1,Brasil,30,Número-índice,0.0000000081223,1,Brasil,2266,IPCA - Número-índice (base: dezembro de 1993 =...,198001,janeiro 1980
2,1,Brasil,30,Número-índice,0.0000000084973,1,Brasil,2266,IPCA - Número-índice (base: dezembro de 1993 =...,198002,fevereiro 1980
3,1,Brasil,30,Número-índice,0.0000000090104,1,Brasil,2266,IPCA - Número-índice (base: dezembro de 1993 =...,198003,março 1980
4,1,Brasil,30,Número-índice,0.0000000094867,1,Brasil,2266,IPCA - Número-índice (base: dezembro de 1993 =...,198004,abril 1980


In [36]:
df[['D4C', 'D4N']].drop_duplicates().sort_values('D4C')

,D4C,D4N
85,101448,1107097.Capa de filé
102,101466,1108072.Peixe - curimatã
381,101642,6301004.Produto para barba
382,101644,6301006.Produto para pele
17,107608,1102029.Massa semipreparada
...,...,...
447,7787,91.Comunicação
448,7788,9101.Comunicação
449,7789,9101001.Correio
454,7794,9101019.Aparelho telefônico


In [44]:
url_2 = "https://apisidra.ibge.gov.br/values/t/7060/n1/all/v/all/p/202001?formato=json"
df_2 = pd.DataFrame(requests.get(url_2).json()[1:])
df_2[['D2C','D2N']].drop_duplicates()


,D2C,D2N
0,63,IPCA - Variação mensal
1,69,IPCA - Variação acumulada no ano
2,2265,IPCA - Variação acumulada em 12 meses
3,66,IPCA - Peso mensal


In [34]:
df.groupby('D4N')['D4C'].unique()

D4N
1.Alimentação e bebidas                                       [7170]
11.Alimentação no domicílio                                   [7171]
1101.Cereais, leguminosas e oleaginosas                       [7172]
1101002.Arroz                                                 [7173]
1101051.Feijão - mulatinho                                    [7175]
                                                              ...   
9101018.Acesso à internet                                   [107688]
9101019.Aparelho telefônico                                   [7794]
9101115.Serviços de streaming                                [47671]
9101116.Combo de telefonia, internet e tv por assinatura     [47672]
Índice geral                                                  [7169]
Name: D4C, Length: 457, dtype: object